# 01 — ETL session và chronological split

Pipeline Coveo v1 — mọi output được version hóa và không ghi đè.

In [ ]:
%pip install -q -e ".[coveo]"

In [ ]:
from pathlib import Path
import os, json, yaml

def find_repo():
    here = Path.cwd().resolve()
    for root in (here, *here.parents):
        if (root / 'pyproject.toml').exists(): return root
    raise FileNotFoundError('Không tìm thấy pyproject.toml')

REPO = find_repo()
os.chdir(REPO)
cfg = yaml.safe_load((REPO / 'configs/coveo.yaml').read_text(encoding='utf-8'))
PROFILE = os.getenv('COVEO_PROFILE', cfg['project']['profile'])
print('repo=', REPO, 'profile=', PROFILE)

In [ ]:
from datn.data.coveo import CoveoPrepareConfig, prepare_coveo_dataset
data_cfg = cfg['data']
params = CoveoPrepareConfig(
    dataset_version=Path(cfg['paths']['processed_dir']).name,
    train_ratio=data_cfg['train_ratio'], valid_ratio=data_cfg['valid_ratio'],
    min_session_events=data_cfg['min_session_events'],
    max_sessions=data_cfg[f'max_sessions_{PROFILE}'],
    include_search_clicks=data_cfg['include_search_clicks'], seed=cfg['project']['seed'])
manifest = prepare_coveo_dataset(REPO / cfg['paths']['raw_dir'], REPO / cfg['paths']['processed_dir'], params)
display(manifest['counts'], manifest['split_time_ranges'])

Split theo thời gian kết thúc session toàn cục. Validation/test là session tương lai; event cuối mỗi session là ground truth, phần trước là context.